# 01 – Explore the sources

Exploration only: this notebook is **not** part of the pipeline. It is the evidence behind the
decisions in `docs/decisions.md` (D1–D13). Each section ends with a short **Decision** note that
names the decision the result supports, so a reviewer can follow the reasoning from data to design.

| Section | Question | Supports |
|---|---|---|
| 1. Service Points v2 | Do our four filters give the 51 Lucerne rail stops? | D3 |
| 2. SwissMetNet stations | Which weather station does each stop get, and why the 300 m rule? | D4 |
| 3. Ist-Daten v2, one day | Format, volume, availability, statuses, stop roles, table key, cancellations, delays | D2, D5–D11 |
| 4. PostgreSQL | Does the connection from `.env` work? | – |
| 5. Ist-Daten archive | Can history beyond 50 days come from the monthly archive? | D13 |

Downloaded files go to `data/<source>/`, which is ignored by Git. Sections 1–4 download what they need and
run with **Restart & Run All**; the large Ist-Daten file is reused from `data/istdaten/` if it is already there.
Section 5 needs one archive ZIP that has to be **downloaded by hand first** (see the section).

In [1]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Works whether Jupyter was started in the repo root or in notebooks/
REPO = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = REPO / "data"
# Downloaded source files, one folder per source. Ignored by Git.
RAW_SP = DATA / "service_points"
RAW_METEO = DATA / "meteoswiss"
RAW_IST = DATA / "istdaten"
RAW_ARCHIVE = DATA / "istdaten_archive"
for folder in (RAW_SP, RAW_METEO, RAW_IST, RAW_ARCHIVE):
    folder.mkdir(parents=True, exist_ok=True)
load_dotenv(REPO / ".env")

DAY = "2026-10-08"   # operating day to explore (a normal Thursday)


def download(url: str, target: Path) -> Path:
    """Download url to target, streaming to disk. Skips if the file is already there."""
    if target.exists():
        print(f"already downloaded: {target.name} ({target.stat().st_size / 1e6:.1f} MB)")
        return target
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(target, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print(f"downloaded: {target.name} ({target.stat().st_size / 1e6:.1f} MB)")
    return target


def detect_separator(path: Path) -> str:
    """Look at the header line and return ';' or ','."""
    with open(path, encoding="utf-8-sig") as f:
        header = f.readline()
    print("header:", header.strip()[:200])
    return ";" if header.count(";") > header.count(",") else ","

## 1. Service Points v2 – the stop list

The service points file lists every stop in Switzerland, including bus stops, boat landings and
cable cars. We keep a stop if it is in canton LU, is a stop point, has coordinates, and is served by
trains.

In [2]:
SERVICE_POINTS_URL = (
    "https://data.opentransportdata.swiss/dataset/39e5f264-257a-4f3f-bccc-5322c37058c5/"
    "resource/738a4c92-5bb7-48ea-848c-f28e668520da/download/actual-date-swiss-service-point.csv"
)
sp_path = download(SERVICE_POINTS_URL, RAW_SP / "actual-date-swiss-service-point.csv")
sp_sep = detect_separator(sp_path)

# dtype=str: read everything as text first, so nothing is silently converted
sp = pd.read_csv(sp_path, sep=sp_sep, dtype=str, encoding="utf-8-sig")
print(f"separator: {sp_sep!r}   rows: {len(sp):,}   columns: {len(sp.columns)}")
sp.head(3)

already downloaded: actual-date-swiss-service-point.csv (26.6 MB)
header: numberShort;uicCountryCode;sloid;number;checkDigit;globalId;validFrom;validTo;designationOfficial;designationLong;abbreviation;operatingPoint;operatingPointWithTimetable;stopPoint;stopPointType;freigh
separator: ';'   rows: 60,359   columns: 56


,numberShort,uicCountryCode,sloid,number,checkDigit,globalId,validFrom,validTo,designationOfficial,designationLong,...,businessOrganisationDescriptionEn,fotComment,lv95East,lv95North,wgs84East,wgs84North,height,creationDate,editionDate,status
0,22001,13,ch:1:sloid:1322001,1322001,2,NaN,2020-09-01,9999-12-31,Antronapiana,NaN,...,Autoservizi Comazzi S.R.L.,NaN,2652235.0,1101179.0,8.11362026337,46.06011993515,0.0,2017-11-09 11:53:05,2024-04-08 09:26:05,VALIDATED
1,22002,13,ch:1:sloid:1322002,1322002,0,NaN,2023-02-23,9999-12-31,Anzola d'Ossola,NaN,...,Autoservizi Comazzi S.R.L.,NaN,2670286.0,1093551.0,8.34571518202,45.98986941001,0.0,2017-11-09 11:53:05,2024-04-08 09:26:05,VALIDATED
2,22003,13,ch:1:sloid:1322003,1322003,8,NaN,2020-09-01,9999-12-31,Baceno,NaN,...,Autoservizi Comazzi S.R.L.,NaN,2667897.0,1123722.0,8.31925580536,46.26150109397,0.0,2017-11-09 11:53:05,2024-04-08 09:26:05,VALIDATED


In [3]:
# How are the filter columns actually written? (true/True/1, LU, TRAIN|BUS ...)
for col in ["cantonAbbreviation", "stopPoint", "hasGeolocation", "meansOfTransport"]:
    print(f"--- {col}")
    print(sp[col].value_counts(dropna=False).head(8).to_string())

--- cantonAbbreviation
cantonAbbreviation
NaN    29599
BE      4136
VD      3095
ZH      2950
VS      2785
GR      2443
TI      2105
SG      1763
--- stopPoint
stopPoint
true     54666
false     5693
--- hasGeolocation
hasGeolocation
true     58410
false     1949
--- meansOfTransport
meansOfTransport
BUS          44052
UNKNOWN       5878
NaN           5693
TRAIN         1726
BUS|TRAM       866
CHAIRLIFT      594
CABLE_CAR      524
BOAT           380


In [4]:
# The four documented filters, applied in code
is_true = lambda s: s.str.lower() == "true"

stops = sp[
    (sp["cantonAbbreviation"] == "LU")
    & is_true(sp["stopPoint"])
    & is_true(sp["hasGeolocation"])
    & sp["meansOfTransport"].str.contains("TRAIN", na=False)
].copy()

print(f"stops after filters: {len(stops)}  (expected: 51)")
print(f"unique numbers:      {stops['number'].nunique()}")
stops[["number", "designationOfficial", "meansOfTransport", "wgs84North", "wgs84East", "height"]] \
    .sort_values("designationOfficial").reset_index(drop=True)

stops after filters: 51  (expected: 51)
unique numbers:      51


,number,designationOfficial,meansOfTransport,wgs84North,wgs84East,height
0,8502025,Baldegg,TRAIN,47.18568895917,8.28023150834,470.0
1,8516353,Baldegg Kloster,TRAIN,47.18061602969,8.28059754295,480.0
2,8502023,Ballwil,TRAIN,47.15376765531,8.31689880205,515.0
3,8502002,Brittnau-Wikon,TRAIN,47.26266615496,7.95704084851,444.0
4,8516350,Buchrain,TRAIN,47.09364991511,8.35665850186,419.0
5,8502004,Dagmersellen,TRAIN,47.21755794081,7.97902592247,472.0
6,8502200,Ebikon,TRAIN,47.08426271808,8.34235825617,421.0
7,8502021,Emmenbrücke,TRAIN,47.07060149967,8.28256536801,438.0
8,8502028,Emmenbrücke Gersag,TRAIN,47.07831417825,8.27682007954,450.0
9,8502012,Emmenbrücke Kapf,TRAIN,47.09090997877,8.27319059247,484.0


> **Decision D3 · Filters in code, never from a GUI export.** The four filters give exactly 51 stops,
> all with a unique `number`. The pipeline applies these same filters to the full file and fails if
> the result is not 51, so a change in the source shows up as an error instead of a silent shift.
> Notes from the value counts above: booleans are the strings `true`/`false`, and `meansOfTransport`
> can hold several values (`BUS|TRAM`), so the train filter uses *contains* `TRAIN`, not equality.

## 2. SwissMetNet stations – which weather belongs to which stop?

MeteoSwiss publishes the list of automatic weather stations (SwissMetNet) with coordinates and
altitude. Each stop gets the nearest station, but stations more than 300 m higher or lower than the
stop are skipped: at that difference temperature, snow and wind can be very different, even if the
station is close. The last column shows which stops the altitude rule actually changes.

In [5]:
import numpy as np

META_URL = "https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/ogd-smn_meta_stations.csv"
meta_path = download(META_URL, RAW_METEO / "ogd-smn_meta_stations.csv")
# MeteoSwiss CSVs use ';' and Windows-1252 encoding (umlauts break with utf-8)
smn = pd.read_csv(meta_path, sep=";", encoding="cp1252")
print(f"SwissMetNet stations: {len(smn)}")
smn[["station_abbr", "station_name", "station_canton", "station_height_masl",
     "station_coordinates_wgs84_lat", "station_coordinates_wgs84_lon"]].head()

already downloaded: ogd-smn_meta_stations.csv (0.2 MB)
SwissMetNet stations: 159


,station_abbr,station_name,station_canton,station_height_masl,station_coordinates_wgs84_lat,station_coordinates_wgs84_lon
0,ABO,Adelboden,BE,1321.0,46.491703,7.560703
1,AEG,Oberägeri,ZG,724.0,47.133636,8.608206
2,AIG,Aigle,VD,381.0,46.326647,6.924472
3,ALT,Altdorf,UR,438.0,46.887069,8.621894
4,AND,Andeer,GR,987.0,46.610139,9.431981


In [6]:
MAX_HEIGHT_DIFF_M = 300


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between points given in degrees (works on numpy arrays)."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))


stop_lat = stops["wgs84North"].astype(float).to_numpy()[:, None]
stop_lon = stops["wgs84East"].astype(float).to_numpy()[:, None]
stop_h = stops["height"].astype(float).to_numpy()[:, None]
stn_lat = smn["station_coordinates_wgs84_lat"].astype(float).to_numpy()[None, :]
stn_lon = smn["station_coordinates_wgs84_lon"].astype(float).to_numpy()[None, :]
stn_h = smn["station_height_masl"].astype(float).to_numpy()[None, :]

dist = haversine_km(stop_lat, stop_lon, stn_lat, stn_lon)       # 51 stops x all stations
height_diff = np.abs(stop_h - stn_h)
allowed = np.where(height_diff <= MAX_HEIGHT_DIFF_M, dist, np.inf)

best = allowed.argmin(axis=1)          # nearest station within the altitude limit
plain = dist.argmin(axis=1)            # nearest station without the rule, for comparison
rows = np.arange(len(stops))

mapping = pd.DataFrame({
    "number": stops["number"].to_numpy(),
    "stop": stops["designationOfficial"].to_numpy(),
    "stop_height_m": stops["height"].astype(float).round().astype(int).to_numpy(),
    "station": smn["station_abbr"].to_numpy()[best],
    "distance_km": dist[rows, best].round(1),
    "height_diff_m": height_diff[rows, best].round().astype(int),
    "nearest_without_rule": smn["station_abbr"].to_numpy()[plain],
}).sort_values(["station", "distance_km"]).reset_index(drop=True)

print("stations used:", sorted(mapping["station"].unique()), " (decided: CHZ, EGO, LUZ, MOA, SPF)")
print("stops per station:")
print(mapping["station"].value_counts().to_string())
print(f"\nlargest stop-to-station distance: {mapping['distance_km'].max()} km")
changed_by_rule = mapping[mapping["station"] != mapping["nearest_without_rule"]]
print(f"stops where the 300 m rule changes the station: {len(changed_by_rule)}")
changed_by_rule

stations used: ['CHZ', 'EGO', 'LUZ', 'MOA', 'SPF']  (decided: CHZ, EGO, LUZ, MOA, SPF)
stops per station:
station
LUZ    20
EGO    15
MOA     8
SPF     6
CHZ     2

largest stop-to-station distance: 14.0 km
stops where the 300 m rule changes the station: 5


,number,stop,stop_height_m,station,distance_km,height_diff_m,nearest_without_rule
16,8516354,Wolhusen Weid,601,EGO,13.6,79,NAP
32,8508218,Malters,496,LUZ,9.0,42,PIL
36,8508217,Schachen LU,518,LUZ,12.0,64,PIL
49,8508215,Wolhusen,565,SPF,13.1,179,NAP
50,8508216,Werthenstein,553,SPF,14.0,191,NAP


> **Decision D4 · Nearest station, max 300 m altitude difference.** Result: exactly the decided
> 5 stations (CHZ, EGO, LUZ, MOA, SPF), at most 14 km from a stop. One of them (CHZ, Cham) is in
> canton Zug: the nearest suitable station doesn't have to be in Lucerne. Without the rule, 5 stops
> (Malters, Schachen LU, Wolhusen, Wolhusen Weid, Werthenstein) would get the mountain stations
> Pilatus (PIL) or Napf (NAP), whose weather says little about a valley station. The distance
> column is kept in the stop table, so analysts can judge how representative the station is for
> each stop (a documented limitation: a local shower can miss the station).

In [7]:
# Weather timestamps: format and range of the hourly file the weather ingestion will read
WEATHER_URL = "https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn/luz/ogd-smn_luz_h_recent.csv"
w_path = download(WEATHER_URL, RAW_METEO / "ogd-smn_luz_h_recent.csv")
weather = pd.read_csv(w_path, sep=";", encoding="cp1252", dtype={"reference_timestamp": str})
ts = pd.to_datetime(weather["reference_timestamp"], format="%d.%m.%Y %H:%M")
print(f"rows: {len(weather):,}   columns: {len(weather.columns)}   first: {ts.min()}   last: {ts.max()}")
print("hours on", DAY, ":", int((ts.dt.strftime("%Y-%m-%d") == DAY).sum()), "(expected 24)")

already downloaded: ogd-smn_luz_h_recent.csv (0.9 MB)
rows: 6,888   columns: 36   first: 2026-01-01 00:00:00   last: 2026-10-14 23:00:00
hours on 2026-10-08 : 24 (expected 24)


> The `recent` hourly file runs from 1 January to yesterday (last value 2026-10-14 23:00 when run on
> 15 October), with all 24 hours for our test day. So the weather for day D is available on D+1,
> like Ist-Daten, and one daily run can load both.
>
> **To verify before joining (weather ingestion, Emre):** MeteoSwiss documents its timestamps as UTC,
> while Ist-Daten times are Swiss local time. The join must convert one to the other, otherwise every
> stop event is matched to weather one or two hours off, and the daylight-saving switch on
> 25 October 2026 makes it worse. Add the source of this to `docs/data_sources.md`.

## 3. Ist-Daten v2 – one operating day

Each daily file has its own resource ID in the URL, so the address **cannot be built from the date
alone**. The portal's catalogue API did not return JSON reliably, so the cell below reads the
download link from the dataset page itself. The ingestion script (`ingestion/check_source.py`) uses
the same lookup.

The file covers all of Switzerland (about 2.6 million rows, 658 MB), so the first download takes a
moment.

In [8]:
import re
from urllib.parse import urljoin

ist_path = RAW_IST / f"{DAY}_istdaten.csv"
DATASET_PAGE = "https://data.opentransportdata.swiss/dataset/ist-daten-v2"

if not ist_path.exists():
    page = requests.get(DATASET_PAGE, timeout=60)
    page.raise_for_status()

    # every daily file appears on the page as a link ending in /download/<date>_istdaten.csv
    all_days = sorted(set(re.findall(r"/download/(\d{4}-\d{2}-\d{2})_istdaten\.csv", page.text, flags=re.I)))
    print(f"daily files listed: {len(all_days)}")
    if all_days:
        print("oldest / newest:", all_days[0], "/", all_days[-1])

    links = re.findall(rf'href="([^"]*/download/{DAY}_istdaten\.csv)"', page.text, flags=re.I)
    assert links, f"no download link for {DAY} on the dataset page (status {page.status_code})"
    download(urljoin(DATASET_PAGE, links[0]), ist_path)

ist_sep = detect_separator(ist_path)
print("separator:", repr(ist_sep))

header: BETRIEBSTAG;FAHRT_BEZEICHNER;BETREIBER_ID;BETREIBER_ABK;BETREIBER_NAME;PRODUKT_ID;LINIEN_ID;LINIEN_TEXT;UMLAUF_ID;VERKEHRSMITTEL_TEXT;ZUSATZFAHRT_TF;FAELLT_AUS_TF;BPUIC;HALTESTELLEN_NAME;ANKUNFTSZEIT;
separator: ';'


In [9]:
# Availability: how many days the page keeps, and when the newest file was published.
# The Last-Modified time tells us when to schedule the daily run.
# A HEAD request is refused with 403 by this server (first run on 15 Oct), so we use GET with
# stream=True: it reads only the response headers and closes before downloading the 658 MB body.
page = requests.get(DATASET_PAGE, timeout=60)
page.raise_for_status()
days = sorted(set(re.findall(r"/download/(\d{4}-\d{2}-\d{2})_istdaten\.csv", page.text, flags=re.I)))
print(f"daily files on the page: {len(days)}   oldest: {days[0]}   newest: {days[-1]}")

for day in days[-3:]:
    link = re.findall(rf'href="([^"]*/download/{day}_istdaten\.csv)"', page.text, flags=re.I)[0]
    with requests.get(urljoin(DATASET_PAGE, link), stream=True, allow_redirects=True, timeout=60) as r:
        print(f"{day}: HTTP {r.status_code}, Last-Modified {r.headers.get('Last-Modified')}, "
              f"size {int(r.headers.get('Content-Length', 0)) / 1e6:.0f} MB")

daily files on the page: 50   oldest: 2026-08-26   newest: 2026-10-14
2026-10-12: HTTP 200, Last-Modified Tue, 13 Oct 2026 15:21:03 GMT, size 660 MB
2026-10-13: HTTP 200, Last-Modified Thu, 15 Oct 2026 03:11:59 GMT, size 662 MB
2026-10-14: HTTP 200, Last-Modified Thu, 15 Oct 2026 03:14:03 GMT, size 661 MB


> **Decisions D8 and D11 · Check the source first, schedule after publication.** The page is a rolling
> window of 50 daily files: on 11 October the oldest was 2026-08-22, on 15 October 2026-08-26. Older
> days, including about one year of history, have to come from the monthly archives, so the backfill
> needs a second download path (to check). The Last-Modified times show when a day's file appears;
> the Kestra schedule (09:00 placeholder) should start after that, and `check_transport_source`
> retries if the file is late.
>
> **For `check_source.py`:** the server refused `HEAD` requests with 403, so the source check can't
> copy the course's `check_source.py`, which uses `HEAD`. Finding the link on the dataset page is
> already the check that the file exists; a `GET` with `stream=True` confirms it is downloadable.

In [10]:
# Read in chunks so the full Swiss file never has to fit in memory.
# Keep only rail rows at our stops; count product types over the whole file on the way.
# PRODUKT_ID is compared case-insensitively: the file has both "Bus" and "BUS" (see output),
# so a future "ZUG" must not be dropped silently.
stop_numbers = set(stops["number"])
product_counts = pd.Series(dtype="int64")
total_rows = 0
kept = []

for chunk in pd.read_csv(ist_path, sep=ist_sep, dtype=str, encoding="utf-8-sig", chunksize=200_000):
    total_rows += len(chunk)
    product_counts = product_counts.add(chunk["PRODUKT_ID"].value_counts(dropna=False), fill_value=0)
    is_rail = chunk["PRODUKT_ID"].str.lower() == "zug"
    kept.append(chunk[is_rail & chunk["BPUIC"].isin(stop_numbers)])

ist = pd.concat(kept, ignore_index=True)
print(f"rows in file: {total_rows:,}")
print(f"Lucerne rail rows: {len(ist):,}  ({len(ist) / total_rows * 100:.2f} % of the file)")
print()
print("PRODUKT_ID values in the whole file:")
print(product_counts.astype(int).sort_values(ascending=False).to_string())

rows in file: 2,589,845
Lucerne rail rows: 5,532  (0.21 % of the file)

PRODUKT_ID values in the whole file:
PRODUKT_ID
Bus            2138294
Tram            248342
Zug             180322
BUS              15457
Metro             4864
Zahnradbahn       1763
Schiff             792
NaN                 11


> **Decisions D5 and D6 · Batch, pull, full load per day, filtered at ingestion.** One file is one
> complete operating day, published once a day: a full load per period, not change data capture.
> Our stops are about 0.2 % of the file, and a year of full files would be about 240 GB. So ingestion
> keeps only our rows, with **every column unchanged**; cleaning happens later. In the final version
> the full file goes to GCS (as Parquet) as the raw zone.

In [22]:
# v2 check: some operators deliver the new SLOID (e.g. ch:1:sloid:5000) in the BPUIC field.
# Are there rail rows at our stops that only the SLOID would find, i.e. that our BPUIC filter misses?
our_sloids = set(stops["sloid"])
missed = 0
for chunk in pd.read_csv(ist_path, sep=ist_sep, dtype=str, encoding="utf-8-sig", chunksize=200_000):
    rail = chunk["PRODUKT_ID"].str.lower() == "zug"
    missed += int((rail & chunk["SLOID"].isin(our_sloids) & ~chunk["BPUIC"].isin(stop_numbers)).sum())
print("rail rows at our stops (by SLOID) that the BPUIC filter misses:", missed)

rail rows at our stops (by SLOID) that the BPUIC filter misses: 0


> **Decision D6 · The BPUIC filter is enough.** Ist-Daten v2 added operators that deliver the SLOID
> in the `BPUIC` field. On 8 October no rail row at our stops is missed by matching `BPUIC` against
> the service point numbers (and none on 29 September, section 5). If this ever changes, the
> ingestion filter matches on `BPUIC` **or** `SLOID`.

In [11]:
# Columns and one example row
print(list(ist.columns))
ist.iloc[0]

['BETRIEBSTAG', 'FAHRT_BEZEICHNER', 'BETREIBER_ID', 'BETREIBER_ABK', 'BETREIBER_NAME', 'PRODUKT_ID', 'LINIEN_ID', 'LINIEN_TEXT', 'UMLAUF_ID', 'VERKEHRSMITTEL_TEXT', 'ZUSATZFAHRT_TF', 'FAELLT_AUS_TF', 'BPUIC', 'HALTESTELLEN_NAME', 'ANKUNFTSZEIT', 'AN_PROGNOSE', 'AN_PROGNOSE_STATUS', 'ABFAHRTSZEIT', 'AB_PROGNOSE', 'AB_PROGNOSE_STATUS', 'DURCHFAHRT_TF', 'SLOID']


BETRIEBSTAG                                 08.10.2026
FAHRT_BEZEICHNER             ch:1:sjyid:100001:173-001
BETREIBER_ID                                     85:11
BETREIBER_ABK                                      SBB
BETREIBER_NAME         Schweizerische Bundesbahnen SBB
PRODUKT_ID                                         Zug
LINIEN_ID                                          173
LINIEN_TEXT                                         EC
UMLAUF_ID                                          NaN
VERKEHRSMITTEL_TEXT                                 EC
ZUSATZFAHRT_TF                                   false
FAELLT_AUS_TF                                    false
BPUIC                                          8505000
HALTESTELLEN_NAME                               Luzern
ANKUNFTSZEIT                          08.10.2026 16:05
AN_PROGNOSE                        08.10.2026 16:05:10
AN_PROGNOSE_STATUS                                REAL
ABFAHRTSZEIT                          08.10.2026 16:18
AB_PROGNOS

In [12]:
# Do all 51 stops appear? Stops with zero rows need an explanation (no service that day? wrong ID?)
per_stop = ist.groupby("BPUIC").size()
check = stops[["number", "designationOfficial"]].assign(
    rows=lambda d: d["number"].map(per_stop).fillna(0).astype(int)
).sort_values("rows", ascending=False)

print(f"stops with rows: {(check['rows'] > 0).sum()} of {len(check)}")
print("stops without rows:", check.loc[check["rows"] == 0, "designationOfficial"].tolist())
check.reset_index(drop=True)

stops with rows: 51 of 51
stops without rows: []


,number,designationOfficial,rows
0,8505000,Luzern,842
1,8508215,Wolhusen,247
2,8502007,Sursee,236
3,8502021,Emmenbrücke,208
4,8502028,Emmenbrücke Gersag,180
5,8508319,Horw,179
6,8516351,Kriens Mattenhof,179
7,8508321,Luzern Allmend/Messe,178
8,8502009,Sempach-Neuenkirch,123
9,8502020,Rothenburg Station,123


In [13]:
# Flags, operators and operating day: which exact strings does the script have to expect?
for col in ["BETRIEBSTAG", "FAELLT_AUS_TF", "ZUSATZFAHRT_TF", "DURCHFAHRT_TF", "BETREIBER_ABK", "VERKEHRSMITTEL_TEXT"]:
    print(f"--- {col}")
    print(ist[col].value_counts(dropna=False).head(10).to_string())

--- BETRIEBSTAG
BETRIEBSTAG
08.10.2026    5532
--- FAELLT_AUS_TF
FAELLT_AUS_TF
false    4974
true      558
--- ZUSATZFAHRT_TF
ZUSATZFAHRT_TF
false    5458
true       74
--- DURCHFAHRT_TF
DURCHFAHRT_TF
false    5532
--- BETREIBER_ABK
BETREIBER_ABK
SBB        3322
BLS-bls    1301
ZB          810
SOB-sob      99
--- VERKEHRSMITTEL_TEXT
VERKEHRSMITTEL_TEXT
S     4441
RE     673
IR     365
PE      34
IC      17
EC       2


### Cancellations: why about 10 %?

558 of 5,532 rows are cancelled (`FAELLT_AUS_TF = true`), far more than everyday disruption. If most of
them sit on one line or a few stops, they are probably planned works (trains replaced by buses).
Planned closures are not caused by weather, so they would hide any weather effect.

In [14]:
cancelled = (ist["FAELLT_AUS_TF"] == "true").rename("cancelled")

print("Arrival status by cancellation:")
print(pd.crosstab(cancelled, ist["AN_PROGNOSE_STATUS"].fillna("(no scheduled arrival)")).to_string())
print()
print("Departure status by cancellation:")
print(pd.crosstab(cancelled, ist["AB_PROGNOSE_STATUS"].fillna("(no scheduled departure)")).to_string())
print()
print("Cancelled rows by operator and line:")
print(ist[cancelled].groupby(["BETREIBER_ABK", "LINIEN_TEXT"]).size().sort_values(ascending=False).head(15).to_string())
print()
print("Cancelled rows by stop:")
print(ist[cancelled].groupby("HALTESTELLEN_NAME").size().sort_values(ascending=False).head(15).to_string())

Arrival status by cancellation:
AN_PROGNOSE_STATUS  (no scheduled arrival)  PROGNOSE  REAL  UNBEKANNT
cancelled                                                            
False                                  546        40  4388          0
True                                     0         0     0        558

Departure status by cancellation:
AB_PROGNOSE_STATUS  (no scheduled departure)  PROGNOSE  REAL  UNBEKANNT
cancelled                                                              
False                                    544        40  4390          0
True                                       0         0     0        558

Cancelled rows by operator and line:
BETREIBER_ABK  LINIEN_TEXT
SBB            S9             496
BLS-bls        RE7             21
SBB            S1              16
BLS-bls        S6              10
SBB            S29              7
               IR27             4
               IR15             3
               IR70             1

Cancelled rows by stop:
HAL

> **Finding (8 October):**
>
> 1. **`UNBEKANNT` means cancelled.** All 558 cancelled rows have status `UNBEKANNT`, and no train
>    that ran has it. It is not a separate data-quality problem but the status of cancelled trains;
>    the 22 cancelled rows at the start of a journey carry it too, even without a scheduled arrival.
> 2. **One line explains almost all cancellations.** 496 of 558 (89 %) are on SBB **S9**, spread
>    evenly over its stops in the canton (35–37 each, from Emmenbrücke to Mosen): the whole line was cancelled for
>    the day, most likely a planned closure with replacement buses. The other 62 cancellations are
>    about 1 % of the rows, an everyday level.
>
> **Consequence:** Ist-Daten has no flag for *planned* versus *unplanned* cancellations. A planned
> closure is not caused by weather, but would count as dozens of weather-related cancellations.
> Proposed rule (to agree): mark a **line on a day** as a closure when nearly all its trips are
> cancelled, keep those rows with a status, and leave them out of the weather comparison. Run this
> notebook with `DAY` set to two or three other dates first, to see how often it happens.

In [15]:
# REAL vs PROGNOSE vs empty, separately for arrivals and departures.
# Only rows that HAVE a scheduled time count: no scheduled arrival = journey starts here (origin).
def status_share(scheduled: str, status: str) -> pd.Series:
    rows = ist[ist[scheduled].notna()]
    return rows[status].fillna("(empty)").value_counts(normalize=True).mul(100).round(2)

print("Arrivals   (% of rows with a scheduled arrival):")
print(status_share("ANKUNFTSZEIT", "AN_PROGNOSE_STATUS").to_string())
print()
print("Departures (% of rows with a scheduled departure):")
print(status_share("ABFAHRTSZEIT", "AB_PROGNOSE_STATUS").to_string())

Arrivals   (% of rows with a scheduled arrival):
AN_PROGNOSE_STATUS
REAL         88.40
UNBEKANNT    10.80
PROGNOSE      0.81

Departures (% of rows with a scheduled departure):
AB_PROGNOSE_STATUS
REAL         88.51
UNBEKANNT    10.69
PROGNOSE      0.81


> **Decision D7 · Flag rows, don't drop them.** About 88 % of times are `REAL` (measured). `UNBEKANNT`
> (about 11 %, not in the documentation) turned out to be exactly the cancelled trains (see the
> finding above). `PROGNOSE` (40 rows, under 1 %) is an unconfirmed forecast for a train that ran. Rows
> without a measured time stay in the data with a status; their delay is NULL. Section "Flag
> preview" below shows that no row is lost.

In [16]:
# Stop role: which combination of scheduled arrival / departure does each row have?
has_arr = ist["ANKUNFTSZEIT"].notna()
has_dep = ist["ABFAHRTSZEIT"].notna()
role = pd.Series("neither", index=ist.index)
role[has_arr & has_dep] = "intermediate"
role[~has_arr & has_dep] = "origin"
role[has_arr & ~has_dep] = "terminus"
print(role.value_counts().to_string())
print()
print("Luzern (8505000):")
print(role[ist["BPUIC"] == "8505000"].value_counts().to_string())

intermediate    4392
terminus         572
origin           568

Luzern (8505000):
origin          365
terminus        364
intermediate    113


> **Decision D2 · Separate arrival and departure delay, plus a stop role.** A row with no scheduled
> arrival is the start of a journey (origin), one with no scheduled departure is the end (terminus).
> This is not missing data. Luzern is mostly origin or terminus, but 113 rows have both, for trains
> that reverse there and continue. The role is derived per row, so it handles both cases.

In [17]:
# Is (operating day, journey, stop) unique? If yes it can be the key of the raw table.
key = ["BETRIEBSTAG", "FAHRT_BEZEICHNER", "BPUIC"]
dupes = ist[ist.duplicated(key, keep=False)].sort_values(key)
print(f"rows sharing a key with another row: {len(dupes)}")
dupes.head(10)

rows sharing a key with another row: 0


,BETRIEBSTAG,FAHRT_BEZEICHNER,BETREIBER_ID,BETREIBER_ABK,BETREIBER_NAME,PRODUKT_ID,LINIEN_ID,LINIEN_TEXT,UMLAUF_ID,VERKEHRSMITTEL_TEXT,...,BPUIC,HALTESTELLEN_NAME,ANKUNFTSZEIT,AN_PROGNOSE,AN_PROGNOSE_STATUS,ABFAHRTSZEIT,AB_PROGNOSE,AB_PROGNOSE_STATUS,DURCHFAHRT_TF,SLOID


> **Decision D10 · Replace the day, keyed on (operating day, journey, stop).** The key is unique in
> the file, so it can be the primary key of the raw table. Because every row belongs to exactly one
> operating day, a rerun deletes that day and inserts it again inside one transaction: no
> duplicates, and corrected source data replaces the old values.

In [18]:
# Do the dates and times parse with the documented formats? Then a first look at delays.
# Delays only from measured (REAL) times of trains that ran.
def parse(col, fmt):
    return pd.to_datetime(ist[col], format=fmt, errors="coerce")

arr_plan = parse("ANKUNFTSZEIT", "%d.%m.%Y %H:%M")
arr_real = parse("AN_PROGNOSE", "%d.%m.%Y %H:%M:%S")
dep_plan = parse("ABFAHRTSZEIT", "%d.%m.%Y %H:%M")
dep_real = parse("AB_PROGNOSE", "%d.%m.%Y %H:%M:%S")

for name, raw, parsed in [("scheduled arrival", "ANKUNFTSZEIT", arr_plan), ("actual arrival", "AN_PROGNOSE", arr_real),
                          ("scheduled departure", "ABFAHRTSZEIT", dep_plan), ("actual departure", "AB_PROGNOSE", dep_real)]:
    print(f"{name:20} failed to parse: {int((ist[raw].notna() & parsed.isna()).sum())}")

ran = ist["FAELLT_AUS_TF"] == "false"
arr_delay = ((arr_real - arr_plan).dt.total_seconds() / 60)[ran & (ist["AN_PROGNOSE_STATUS"] == "REAL")]
dep_delay = ((dep_real - dep_plan).dt.total_seconds() / 60)[ran & (ist["AB_PROGNOSE_STATUS"] == "REAL")]

print()
print(pd.DataFrame({"arrival": arr_delay.describe(percentiles=[0.5, 0.9, 0.99]),
                    "departure": dep_delay.describe(percentiles=[0.5, 0.9, 0.99])}).round(2).to_string())

print()
print("Share of trains at least N minutes late (candidate thresholds):")
print(pd.DataFrame({f">= {t} min": [(arr_delay >= t).mean() * 100, (dep_delay >= t).mean() * 100]
                    for t in [1, 2, 3, 5]}, index=["arrival %", "departure %"]).round(1).to_string())

scheduled arrival    failed to parse: 0
actual arrival       failed to parse: 0
scheduled departure  failed to parse: 0
actual departure     failed to parse: 0

       arrival  departure
count  4388.00    4390.00
mean      1.26       1.85
std       1.93       1.86
min      -3.17      -0.20
50%       0.88       1.47
90%       2.87       3.37
99%       9.67      10.49
max      20.63      20.28

Share of trains at least N minutes late (candidate thresholds):
             >= 1 min  >= 2 min  >= 3 min  >= 5 min
arrival %        46.1      20.5       9.0       3.3
departure %      69.5      32.8      13.1       4.1


> **To agree · Delay threshold and main measure.** SBB counts a train as punctual if it is less than
> 3 minutes late, so "delayed" = **3 minutes or more** keeps our numbers comparable with the official
> figure: 9.0 % of arrivals and 13.1 % of departures on 8 October. Departures are systematically later
> than arrivals (median 1.5 vs 0.9 min) and almost never early, while arrivals can be up to 3 min
> early. This supports keeping the two separate (D2). Since SBB's punctuality figure is based on
> arrivals, arrival delay is the natural main measure, with departure delay as a second one.
> The delay in minutes is stored as well, so a different threshold is only a change in the
> transformation. Negative values are early trains and are kept as they are.

### Flag preview: what the first transformation will produce

Every row gets a status for its arrival and departure time. The table must have the same number of
rows as the input: rows are flagged, never removed.

In [19]:
def time_basis(scheduled: str, status: str) -> pd.Series:
    s = ist[status]
    return pd.Series(np.select(
        [ist[scheduled].isna(), s.eq("REAL"), s.eq("PROGNOSE"), s.eq("UNBEKANNT")],
        ["no scheduled time", "measured", "forecast only", "unknown"],
        default="unexpected value",
    ), index=ist.index)

flags = pd.DataFrame({
    "cancelled": ist["FAELLT_AUS_TF"] == "true",
    "arrival_basis": time_basis("ANKUNFTSZEIT", "AN_PROGNOSE_STATUS"),
    "departure_basis": time_basis("ABFAHRTSZEIT", "AB_PROGNOSE_STATUS"),
})
assert len(flags) == len(ist), "a flag step must never change the row count"
assert not flags.isin(["unexpected value"]).any().any(), "new status value: extend the rule before loading"

print(f"rows in: {len(ist):,}   rows out: {len(flags):,}")
print()
print(pd.crosstab(flags["arrival_basis"], flags["cancelled"], margins=True).to_string())
print()
usable = ((flags["arrival_basis"] == "measured") & ~flags["cancelled"]).sum()
print(f"arrivals usable for delay statistics: {usable:,} of {len(flags):,} rows")

rows in: 5,532   rows out: 5,532

cancelled          False  True   All
arrival_basis                       
forecast only         40     0    40
measured            4388     0  4388
no scheduled time    546    22   568
unknown                0   536   536
All                 4974   558  5532

arrivals usable for delay statistics: 4,388 of 5,532 rows


> Both checks passed: 5,532 rows in and out, no unexpected status. They are the same checks
> `ingestion/validate.py` will run after each load (Decision D8): the row count is unchanged, and a
> status value nobody has seen before stops the run instead of being silently miscounted. The
> crosstab also shows the 22 cancelled trains that were due to *start* at our stops.

## 4. PostgreSQL – does the connection from `.env` work?

In [20]:
url = "postgresql+psycopg2://{u}:{p}@{h}:{port}/{db}".format(
    u=os.environ["POSTGRES_USER"], p=os.environ["POSTGRES_PASSWORD"],
    h=os.environ["POSTGRES_HOST"], port=os.environ["POSTGRES_PORT"], db=os.environ["POSTGRES_DB"],
)
engine = create_engine(url)
with engine.connect() as conn:
    print(conn.execute(text("select version()")).scalar())

PostgreSQL 18.6 (Debian 18.6-1.pgdg12+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit


## 5. Ist-Daten archive – history beyond 50 days

The dataset page only keeps the last 50 daily files (section 3). For older days, opentransportdata.swiss
runs a separate archive: [archive.opentransportdata.swiss](https://archive.opentransportdata.swiss/actual_data_archive.htm)
has one ZIP per month with all daily files of that month, from 2016 to the last complete month.
Since July 2025 there are two ZIPs per month: `ist-daten-YYYY-MM.zip` (old v1 format) and
**`ist-daten-v2-YYYY-MM.zip`**, the same v2 format as our daily files. We only use v2, which starts
on 13 July 2025 (that month's ZIP is smaller), so one year of history is October 2025 to September 2026.

**Before running this section, download the archive file by hand:**

1. Open [archive.opentransportdata.swiss/actual_data_archive.htm](https://archive.opentransportdata.swiss/actual_data_archive.htm) in a browser.
2. Under **2026**, download **`ist-daten-v2-2026-09.zip`** (about 1.4 GB).
3. Save it in `data/istdaten_archive/`, without unpacking it.

The ZIP is downloaded by hand here only because this is a one-off exploration. The backfill script
downloads the archive ZIPs itself: the platform's [terms of use](https://opentransportdata.swiss/en/terms-of-use/)
describe the data as freely available in machine-readable format, and request limits only apply to the
API services, not to file downloads. The script downloads one ZIP at a time, reads its days straight from
the ZIP, and deletes it before the next one.

The cells below check what the backfill loader relies on: one file per day with no gaps, the same
columns as the daily file, one operating day per file, the same filter working on files read straight
from the ZIP (without unpacking about 19 GB), and whether an archive day equals the daily file.

In [21]:
import zipfile

ZIP_PATH = RAW_ARCHIVE / "ist-daten-v2-2026-09.zip"
assert ZIP_PATH.exists(), f"download {ZIP_PATH.name} by hand first (see the instructions above) and save it in data/istdaten_archive/"

# A first look: how many files, how they are named, and whether the columns match the daily file
with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()
    print(f"files in ZIP: {len(names)}   first: {names[0]}   last: {names[-1]}")
    with z.open(names[0]) as f:
        header = f.readline().decode("utf-8-sig").strip()
    print("same columns as the daily file:", header.split(";") == list(ist.columns))

files in ZIP: 30   first: 2026-09-01_IstDaten.csv   last: 2026-09-30_IstDaten.csv
same columns as the daily file: True


In [23]:
import re, time, zipfile

ZIP_PATH = RAW_ARCHIVE / "ist-daten-v2-2026-09.zip"
TEST_DAY = "2026-09-29"          # a normal Tuesday, still in the daily-file window
DAY_FILE = re.compile(r"(\d{4}-\d{2}-\d{2})_istdaten\.csv$", re.I)   # IstDaten / istdaten


def read_our_rows(source):
    """Same filter as for the daily file: rail rows at our 51 stops, every column unchanged.
    Also counts rows at our stops that only the SLOID would catch (v2 check)."""
    kept, total, missed = [], 0, 0
    for chunk in pd.read_csv(source, sep=";", dtype=str, encoding="utf-8-sig", chunksize=200_000):
        total += len(chunk)
        rail = chunk["PRODUKT_ID"].str.lower() == "zug"
        ours = chunk["BPUIC"].isin(stop_numbers)
        kept.append(chunk[rail & ours])
        missed += int((rail & chunk["SLOID"].isin(our_sloids) & ~ours).sum())
    return pd.concat(kept, ignore_index=True), total, missed


with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()
    day_of = {n: m.group(1) for n in names if (m := DAY_FILE.search(n))}
    expected = pd.date_range("2026-09-01", "2026-09-30").strftime("%Y-%m-%d")
    print(f"daily files: {len(day_of)}   other files: {len(names) - len(day_of)}   "
          f"missing days: {sorted(set(expected) - set(day_of.values()))}")

    sizes = [z.getinfo(n).file_size / 1e6 for n in day_of]
    print(f"unpacked size per day: {min(sizes):.0f}–{max(sizes):.0f} MB, month {sum(sizes) / 1e3:.1f} GB")

    name = next(n for n, d in day_of.items() if d == TEST_DAY)
    start = time.perf_counter()
    with z.open(name) as f:                      # read straight from the ZIP, nothing unpacked
        archive_rows, archive_total, archive_missed = read_our_rows(f)

print(f"\n{TEST_DAY} from the ZIP: {archive_total:,} rows, ours {len(archive_rows):,}, "
      f"read in {time.perf_counter() - start:.0f} s")
print("operating days inside the file:", archive_rows["BETRIEBSTAG"].unique().tolist())
print("rows the BPUIC filter misses (SLOID check):", archive_missed)

daily files: 30   other files: 0   missing days: []
unpacked size per day: 433–689 MB, month 18.7 GB

2026-09-29 from the ZIP: 2,614,536 rows, ours 5,419, read in 12 s
operating days inside the file: ['29.09.2026']
rows the BPUIC filter misses (SLOID check): 0


September is still inside the 50-day window of the dataset page, so the same day exists both in the
archive and as a daily file. The next cell downloads the daily file for `TEST_DAY` (about 660 MB) and
compares the two stop event by stop event. Delete both large files from `data/` afterwards.

In [24]:
daily_path = RAW_IST / f"{TEST_DAY}_istdaten.csv"
link = re.findall(rf'href="([^"]*/download/{TEST_DAY}_istdaten\.csv)"', page.text, flags=re.I)[0]
download(urljoin(DATASET_PAGE, link), daily_path)
daily_rows, daily_total, _ = read_our_rows(daily_path)

print(f"rows in file   archive: {archive_total:,}   daily: {daily_total:,}")
print(f"our rows       archive: {len(archive_rows):,}   daily: {len(daily_rows):,}")

key = ["BETRIEBSTAG", "FAHRT_BEZEICHNER", "BPUIC"]
a = archive_rows.set_index(key).sort_index().fillna("")
d = daily_rows.set_index(key).sort_index().fillna("")
print("same stop events (keys):", a.index.equals(d.index))
if a.index.equals(d.index):
    changed = (a != d)
    print("stop events with any different value:", int(changed.any(axis=1).sum()))
    print("differences per column:", changed.sum()[lambda s: s > 0].to_dict())
else:
    print("only in archive:", len(a.index.difference(d.index)), "  only in daily:", len(d.index.difference(a.index)))

downloaded: 2026-09-29_istdaten.csv (664.5 MB)
rows in file   archive: 2,614,536   daily: 2,614,536
our rows       archive: 5,419   daily: 5,419
same stop events (keys): True
stop events with any different value: 0
differences per column: {}


> **Decision D13 · History from the monthly v2 archive.** The September 2026 ZIP holds 30 daily files,
> no missing days, one operating day per file, with the same columns as the daily file. The archive
> file for 2026-09-29 is identical to the daily file (2,614,536 rows, 5,419 stop events for our stops,
> 0 differing values), and the BPUIC filter misses no rows (SLOID check: 0). So history can come from
> the archive with the same filter and the same replace-the-day load. Reading one day straight from
> the ZIP takes about 12 s: about 6 min per month, 1.2 h for a year, plus about 16 GB of downloads,
> one ZIP at a time. File names differ in case (`IstDaten` vs `istdaten`), so match case-insensitively.

## Findings for `docs/data_sources.md` (operating days 2026-10-08 and 2026-09-29, run on 2026-10-15)

| Question | Result | Decision |
|---|---|---|
| Separator and encoding | Service points and Ist-Daten: `;`, UTF-8. MeteoSwiss: `;`, Windows-1252 | – |
| Stops after the four filters | 51, all with a unique number | D3 |
| Weather stations | 5 (CHZ, EGO, LUZ, MOA, SPF), max 14 km; the 300 m rule keeps 5 stops off Pilatus and Napf | D4 |
| Weather availability | `recent` hourly file covers 1 Jan to yesterday, 24 values per day | D11 |
| File size and volume | 658 MB, about 2.6 M rows per day; our rail rows 5,532 (0.21 %) | D5, D6 |
| SLOID in the `BPUIC` field (v2) | 0 rail rows at our stops missed by the `BPUIC` filter, on both days | D6 |
| Days kept as daily files | rolling window of 50 days (oldest 2026-08-26 on 15 Oct) | D11 |
| History beyond 50 days | monthly v2 ZIPs from July 2025, downloaded by hand; September: 30 files, one day each, same columns; read from the ZIP in about 12 s per day; archive day identical to the daily file | D13 |
| Publication time of a daily file | yesterday's file is there by about 05:15 Swiss time; files can be republished (13 Oct changed again on 15 Oct) | schedule |
| Source check | server refuses `HEAD` (403): use the page link plus a streamed `GET` | D8 |
| `PRODUKT_ID` values | `Zug` for rail; casing varies (`Bus`, `BUS`), so compare case-insensitively | D6 |
| File names | `istdaten` on the dataset page, `IstDaten` in the archive: match case-insensitively | D13 |
| Flag values | `true` / `false` as text | – |
| Stops without rows | none, all 51 have service | – |
| Time status | `REAL` about 88 %; `UNBEKANNT` about 11 % = exactly the cancelled trains; `PROGNOSE` 40 rows | D7 |
| Stop roles | 4,392 intermediate, 572 terminus, 568 origin; Luzern also has 113 intermediate | D2 |
| Is (operating day, journey, stop) unique? | yes, 0 duplicates | D10 |
| Times parse with documented formats? | yes, 0 failures | – |
| Cancellations | 558 rows; 496 on S9 (whole line, planned closure likely); the rest about 1 % | open: closure rule |
| Delay (REAL, trains that ran) | median 0.9 min arrival, 1.5 min departure; at 3 min or more: 9.0 % / 13.1 % | threshold |
| Weather timestamps | `dd.mm.yyyy HH:MM`, documented as UTC: convert before the join | to verify |